# EP05 — Transfer Learning, BERT & GPT
**Exam Relevance:** 2025 Q5 (MCQ), Q6 (15 marks) | 2024 Q5 (18 marks)

This is the highest-weighted topic in Part 2. Understand it deeply.

Topics:
1. What is transfer learning and why it works
2. Pre-training + fine-tuning paradigm
3. True/False statements (common exam format)
4. BERT architecture and training tasks (MLM + NSP)
5. BERT vs GPT — when to use which

---

## 1. What is Transfer Learning?

Transfer learning = taking a model trained on one task/dataset and reusing (part of) it for a different but related task.

**Why it works:** Early layers of deep networks learn general features (edges in images, syntax in text) that are useful across many tasks. Only later layers are task-specific.

**Three approaches:**
1. **Feature extraction** — freeze all pretrained layers, only train a new head
2. **Fine-tuning** — unfreeze some/all layers and continue training with a small LR
3. **Domain adaptation** — adapt the model to a new domain with unlabelled data

**2024 Q5.3 Answer:** C (Random initialization is NOT a transfer learning method)

**2025 Q5.1 Answer:** B ("Requires large labelled datasets for the target task" is NOT a benefit — transfer learning actually reduces this need)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Visualise feature extraction vs fine-tuning
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

layers = ['Conv1\n(edges)', 'Conv2\n(textures)', 'Conv3\n(shapes)', 'Conv4\n(objects)', 'Head\n(task)']
colors_frozen = ['#4ECDC4'] * 4 + ['#FF6B6B']
colors_finetune = ['#4ECDC4', '#4ECDC4', '#FFE66D', '#FFE66D', '#FF6B6B']

for ax, colors, title in zip(
    axes,
    [colors_frozen, colors_finetune],
    ['Feature Extraction\n(all pretrained layers frozen)', 'Fine-tuning\n(later layers unfrozen)']
):
    bars = ax.barh(layers, [1]*5, color=colors, edgecolor='white', height=0.6)
    ax.set_xlim(0, 1.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')
    for i, (bar, color) in enumerate(zip(bars, colors)):
        label = 'FROZEN' if color == '#4ECDC4' else ('TRAINABLE' if color == '#FFE66D' else 'NEW HEAD')
        ax.text(1.05, bar.get_y() + bar.get_height()/2, label, va='center', fontsize=9)

legend_elements = [
    mpatches.Patch(color='#4ECDC4', label='Frozen (pretrained weights kept)'),
    mpatches.Patch(color='#FFE66D', label='Trainable (fine-tuned)'),
    mpatches.Patch(color='#FF6B6B', label='New head (trained from scratch)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=9)
plt.suptitle('Transfer Learning Strategies', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2. The True/False Statements (2025 Q6.1)

These exact statement types appear every year. Know them cold.

### a) "Using a pretrained model always guarantees better results than training from scratch."
**FALSE.**  
If the source domain is very different from the target domain (negative transfer), pretrained weights can actually hurt. E.g., a model pretrained on English text may not help much for a highly specialised chemistry notation task.

### b) "During fine-tuning, all layers of a pretrained model are always frozen."
**FALSE.**  
There are multiple strategies: freeze all (feature extraction), freeze early layers + unfreeze later ones, or unfreeze everything. The right choice depends on how similar source and target tasks are and how much labelled data you have.

### c) "Fine-tuning requires significantly less computational resources than pretraining."
**TRUE.**  
Pretraining requires training on massive datasets (billions of tokens/images) from scratch. Fine-tuning starts from already-good weights and only trains for a few epochs on a much smaller dataset — orders of magnitude cheaper.

## 3. BERT — Architecture and Training Tasks

**Architecture:** Encoder-only (bidirectional Transformer)
- Sees the entire input sequence at once (left AND right context)
- Good at understanding tasks (classification, NER, QA)
- NOT good at generation (no autoregressive decoding)

### BERT's Two Pre-training Tasks:

**1. Masked Language Modelling (MLM)**
- Randomly mask 15% of input tokens (replace with [MASK])
- Model must predict the original token from context
- Forces the model to understand bidirectional context
- Example: "The cat sat on the [MASK]" → predict "mat"

**2. Next Sentence Prediction (NSP)**
- Given two sentences A and B, predict: is B the actual next sentence after A?
- 50% of pairs are real consecutive sentences, 50% are random
- Forces the model to understand inter-sentence relationships
- Useful for QA, natural language inference tasks

**[CLS] token (2024 Q5.4.3):**
- A special token prepended to every input
- Its final hidden state aggregates information from the entire sequence
- Used as the sequence-level representation for classification tasks

In [ ]:
# Visualise BERT vs GPT architectures conceptually
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# BERT — Bidirectional
ax = axes[0]
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 4)
ax.axis('off')
ax.set_title('BERT — Bidirectional Encoder', fontsize=12, fontweight='bold', color='blue')

tokens = ['[CLS]', 'The', '[MASK]', 'is', 'blue']
for i, tok in enumerate(tokens):
    # Input
    color = '#FF9999' if tok == '[MASK]' else ('#FFD700' if tok == '[CLS]' else '#AED6F1')
    ax.add_patch(plt.Rectangle((i*1.0, 0), 0.9, 0.5, color=color, ec='gray'))
    ax.text(i*1.0+0.45, 0.25, tok, ha='center', va='center', fontsize=8)
    # Output
    ax.add_patch(plt.Rectangle((i*1.0, 3), 0.9, 0.5, color='lightgreen', ec='gray'))
    ax.text(i*1.0+0.45, 3.25, f'h{i}', ha='center', va='center', fontsize=9)
    # Arrow
    ax.annotate('', xy=(i*1.0+0.45, 3), xytext=(i*1.0+0.45, 0.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Bidirectional arrows
for i in range(len(tokens)-1):
    ax.annotate('', xy=(i*1.0+0.9+0.1, 1.7), xytext=(i*1.0+0.45, 1.7),
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.2))
    ax.annotate('', xy=(i*1.0+0.45, 1.7), xytext=(i*1.0+1.0+0.45, 1.7),
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.2))

ax.add_patch(plt.Rectangle((-0.1, 1.1), 5.5, 1.2, color='lightblue', alpha=0.3, ec='blue'))
ax.text(2.7, 1.7, 'Bidirectional\nEncoder', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(2.2, -0.2, '← attends to left AND right context →', ha='center', fontsize=8, color='blue')

# GPT — Autoregressive
ax = axes[1]
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 4)
ax.axis('off')
ax.set_title('GPT — Autoregressive Decoder', fontsize=12, fontweight='bold', color='red')

tokens_gpt = ['<s>', 'A', 'B', 'C', 'D']
outputs_gpt = ['A', 'B', 'C', 'D', 'E']
for i, (inp, out) in enumerate(zip(tokens_gpt, outputs_gpt)):
    ax.add_patch(plt.Rectangle((i*1.0, 0), 0.9, 0.5, color='#AED6F1', ec='gray'))
    ax.text(i*1.0+0.45, 0.25, inp, ha='center', va='center', fontsize=9)
    ax.add_patch(plt.Rectangle((i*1.0, 3), 0.9, 0.5, color='lightyellow', ec='gray'))
    ax.text(i*1.0+0.45, 3.25, out, ha='center', va='center', fontsize=9)
    ax.annotate('', xy=(i*1.0+0.45, 3), xytext=(i*1.0+0.45, 0.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Causal (left-only) arrows
for i in range(1, len(tokens_gpt)):
    for j in range(i):
        alpha = 0.5 if i - j == 1 else 0.15
        ax.annotate('', xy=(i*1.0+0.45, 1.7), xytext=(j*1.0+0.45, 1.7),
                    arrowprops=dict(arrowstyle='->', color='red', lw=0.8, alpha=alpha))

ax.add_patch(plt.Rectangle((-0.1, 1.1), 5.5, 1.2, color='#FFCCCC', alpha=0.3, ec='red'))
ax.text(2.7, 1.7, 'Causal\nDecoder', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(2.2, -0.2, '← only left context (cannot see future tokens) →', ha='center', fontsize=8, color='red')

plt.tight_layout()
plt.show()

## 4. BERT vs GPT — Which to Use?

**2025 Q6.2b asked:** *For text summarisation, which model is more suitable?*

**Answer: GPT** (or any autoregressive decoder / encoder-decoder model)

| Task | Use | Why |
|---|---|---|
| Text classification | BERT | Needs bidirectional understanding |
| Named entity recognition | BERT | Tag each token using full context |
| Question answering | BERT | Find answer span from passage |
| **Text generation** | **GPT** | **Autoregressive — generates token by token** |
| **Text summarisation** | **GPT** | **Generation task — output a new sequence** |
| Translation | Encoder-Decoder (T5, BART) | Needs both understanding and generation |

**Key rule:** BERT = understanding tasks. GPT = generation tasks.

In [ ]:
# Demonstrate fine-tuning concept: feature extraction vs full fine-tuning
# (No actual BERT loading — concept demo with a small proxy model)

torch.manual_seed(42)

class PretrainedEncoder(nn.Module):
    """Simulates a pretrained encoder (proxy for BERT layers)."""
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(64, 64)  # early layer (general features)
        self.layer2 = nn.Linear(64, 64)  # late layer (task-specific features)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.relu(self.layer2(self.relu(self.layer1(x))))

class FineTunedClassifier(nn.Module):
    """Add a task-specific head on top of the encoder."""
    def __init__(self, freeze_encoder=True):
        super().__init__()
        self.encoder = PretrainedEncoder()
        self.head = nn.Linear(64, 2)
        
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
    
    def forward(self, x):
        return self.head(self.encoder(x))

# Count trainable params for each strategy
for strategy, freeze in [('Feature Extraction (freeze encoder)', True), 
                           ('Full Fine-tuning (train everything)', False)]:
    model = FineTunedClassifier(freeze_encoder=freeze)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    print(f"{strategy}:")
    print(f"  Total params:     {total:,}")
    print(f"  Trainable params: {trainable:,}  ({trainable/total*100:.1f}%)")
    print(f"  Frozen params:    {frozen:,}  ({frozen/total*100:.1f}%)")
    print()

## Exam Quick-Reference Summary

**Transfer Learning key facts:**
- NOT a benefit: requires large labelled target dataset (it reduces this need)
- NOT a method: random initialization
- Fine-tuning ≠ always freezing all layers (False!)
- Fine-tuning IS much cheaper than pretraining (True!)

**BERT pre-training tasks (must know both):**
1. **MLM** — predict randomly masked tokens (bidirectional context)
2. **NSP** — predict if sentence B follows sentence A

**[CLS]** = aggregate token for sequence-level tasks

**BERT vs GPT:**
- BERT → understanding (classification, NER, QA)
- GPT → generation (summarisation, translation, chatbots)